In [ ]:
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import pandas as pd
import scipy.io as sio
import json
import sys

import sklearn_extra
#from sklearn_extra.cluster import KMedoids

In [ ]:

#REPO_DIR  = os.path.abspath(os.path.join(os.getcwd(), '/Users/lizkal/Library/CloudStorage/SynologyDrive-Personal/SData_ReachGrasp'))
REPO_DIR  = os.path.abspath(os.path.join(os.getcwd(), r'C:\Users\annas\SynologyDrive\MedUniWien\Projects\NeuroClasp\Students\Liz Kalenteridis\SData_ReachGrasp'))
INPUT_DIR = os.path.join(REPO_DIR, 'data')

subjs = ['sub-01', 'sub-02', 'sub-03', 'sub-04']
data_type = ['emg', 'motion', 'tactile']
devices = ['sessantaquattro','cometa','vicon','cyberglove','tactileglove']
tasks = ['HO','HC','WP','WS','WF','WE','Cyl','Sph','Trid','Thumb','FroRea','ReaCyl','ReaSph','Screw','Pour','EatFruit']

n_subj = len(subjs)
n_tasks = len(tasks)

In [ ]:

nChann = 64; # Number of HD-sEMG channels
fs = 2000; # Sampling freuqency [Hz]
unit_factor = 1e-3; # factor used to convert mV in V
noverlap = 500
nfft = 4001

# pass band frequency
f_pass = 20
wp = f_pass / (fs/2) # in radians
# stop band frequency
f_stop = 500
ws = f_stop / (fs/2) # in radians

bandpass_order = 2

window = signal.windows.hamming(fs)

bp_b, bp_a = signal.butter(bandpass_order, [20, 500], btype = 'bandpass', fs = fs) # does output='sos' belong here?


# notch filter at 50Hz
notch = 50
bw = 0.2 # bandwidth
Q = notch / bw # quality factor, higher = narrower notch

b_n, a_n = signal.iirnotch(notch, Q, fs = fs)

In [ ]:
fs_Vicon = 100 # sampling rate for kinematic data used to determine movement bounds with triggers
event_file = os.path.join(REPO_DIR, 'code/Events_Reach&Grasp.mat')
event_mat = sio.loadmat(event_file, struct_as_record = False, squeeze_me = True)
events = event_mat['Events_ReachGrasp']

trigger_lookup = {}
trigger_lookup_extended = {}
for subj in events.subjects:
    for task in subj.tasks:
        trigger_lookup[(subj.subject_name, task.task_name)] = np.asarray(task.time2cut/fs_Vicon) # convert to index from seconds
        trigger_lookup_extended[(subj.subject_name, task.task_name)] = np.asarray(task.time2cut/fs_Vicon + 0.2) # add 0.2 seconds to the bounds
        # kinematic data sets the bounds based on when the movement visibly start and ends, EMG decomposition may require larger window

In [ ]:
# create dictionary containing data within the trigger bounds unique to each subject and task.
# ex movement_subset['sub-01']['HO']['emg_data'] yields the data for sub-01 during the task HO for when the kinematic data indicates movement start/end times
# ex movement_subset['sub-01']['HO']['time_labels']  yields and array of the specific time values for the movement bounds
movement_subset = {}

for subj in subjs:
    for task in tasks:
        trigger_vals = trigger_lookup_extended[(subj, task)]

        task_file_name = os.path.join(INPUT_DIR, 
                                            subj, 
                                            f'emg/{subj}_task-{task}_acq-sessantaquattro_emg.csv'
                                            ) 
        task_df = pd.read_csv(task_file_name, header = None)
        task_df.columns =  ['time'] + [f'channel{i}' for i in range(1, nChann + 1)] # rename so col 1 is named time, 2 is channel 1... 
        task_df_t = task_df.set_index('time').T # set time as the index and transpose so they become the columns
        task_df_t.reset_index(drop = True, inplace = True)
        #task_df_t = task_df_t[:, 1:]

        col_times = task_df_t.columns.to_numpy(dtype=float)
            
        mask = np.zeros(len(col_times), dtype=bool)
        for start, end in np.array(trigger_vals).reshape(-1, 2):
            mask |= (col_times >= start) & (col_times <= end)
        
        if subj not in movement_subset:
            movement_subset[subj] = {}

        filtered = task_df_t.loc[:, mask]
        
        movement_subset[subj][task]  = {
            'emg_data': filtered.to_numpy(),                        # data within trigger bounds (shape: nChann x nSamples)
            'time_labels': filtered.columns.to_numpy(dtype=float)   # array of the time values for the specific movement and subject within trigger bounds
        }


In [ ]:
emg_data_HO1 = movement_subset['sub-02']['Trid']['emg_data']
plt.figure(figsize=(12, 6))
plt.plot(emg_data_HO1[5,:])
plt.title("Combined EMG Data")
plt.xlabel("Time (samples)")
plt.ylabel("Amplitude")
plt.grid()

emg_raw = emg_data_HO1.copy()

In [ ]:
from scipy.interpolate import interp1d

def remove_spikes(emg, threshold_std=5):
    """
    Remove large spikes via simple thresholding and linear interpolation.
    
    Parameters
    ----------
    emg : ndarray, shape (n_channels, n_samples)
    threshold_std : float
        Number of standard deviations for threshold
    
    Returns
    -------
    emg_clean : ndarray
    spike_mask : ndarray, bool
    """
    emg_clean = emg.copy()
    spike_mask = np.zeros_like(emg, dtype=bool)
    
    for ch in range(emg.shape[0]):
        signal = emg[ch]
        threshold = threshold_std * np.std(signal)
        
        spikes = np.abs(signal) > threshold
        spike_mask[ch] = spikes
        
        if np.any(spikes):
            clean_idx = np.where(~spikes)[0]
            spike_idx = np.where(spikes)[0]
            
            if len(clean_idx) > 1:
                interp = interp1d(clean_idx, signal[clean_idx], 
                                  kind='linear', bounds_error=False, 
                                  fill_value='extrapolate')
                emg_clean[ch, spike_idx] = interp(spike_idx)
    
    return emg_clean, spike_mask


In [ ]:
emg_clean, spikes = remove_spikes(emg_raw, threshold_std=5)

#plot emg before and after spike removal for first channel
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))        
plt.subplot(2, 1, 1)
plt.plot(emg_raw[0], label='Raw EMG', color='red')
plt.title(f'Raw EMG Signal - Channel 0')
plt.xlabel('Samples')
plt.ylabel('Amplitude')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(emg_clean[0], label='Cleaned EMG', color='blue')
plt.title(f'Cleaned EMG Signal - Channel 0 ')
plt.xlabel('Samples')
plt.ylabel('Amplitude')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:


sys.path.append(os.path.abspath(r'C:\Users\annas\SynologyDrive\MedUniWien\Projects\NeuroClasp\Students\Liz Kalenteridis\MotorUnitSuite-main\src'))


# Custom modules
from muniverse.algorithms.decompositionLK import decompose_cbss


#algo_cfg = r"C:\Users\annas\SynologyDrive\MedUniWien\Projects\NeuroClasp\Students\Liz Kalenteridis\MotorUnitSuite-main\src\configs\cbss.json"
with open(r"C:\Users\annas\SynologyDrive\MedUniWien\Projects\NeuroClasp\Students\Liz Kalenteridis\MotorUnitSuite-main\src\configs\cbss.json") as f:
    algo_cfg = json.load(f)["Config"]

#algo_cfg["sil_th"] = 0.7

results, metadata = decompose_cbss(
        data=emg_clean,
        algorithm_config=algo_cfg,
        show_config=False, 
)




FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled
FR peel-off enabled


In [ ]:
results["spikes"].keys()